# Basic RAG - chạy từng bước
Chạy lần lượt từng cell từ trên xuống.

In [1]:
import json
import os
import warnings

from dotenv import load_dotenv
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")
load_dotenv()

INDEX_DIR = "faiss_index"

USER_AGENT environment variable not set, consider setting it to identify your requests.


## Bước 1: Load tài liệu từ web

In [2]:
loader = WebBaseLoader("https://python.langchain.com/docs/get_started/introduction")
documents = loader.load()

print("Number of documents:", len(documents))
for i, doc in enumerate(documents):
    print(f"\nDocument {i + 1}")
    print("Metadata:", doc.metadata)
    print("Content length:", len(doc.page_content))
    print(doc.page_content[:1000])

Number of documents: 1

Document 1
Metadata: {'source': 'https://python.langchain.com/docs/get_started/introduction', 'title': 'Quickstart - Docs by LangChain', 'description': 'Build your first agent in minutes', 'language': 'en'}
Content length: 25001






Quickstart - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationGet startedQuickstartOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middleware

## Bước 2: Chia chunk (1000 ký tự, gối đầu 200)

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print("\nChunk đầu tiên:\n", chunks[0].page_content)

Total chunks: 37

Chunk đầu tiên:
 Quickstart - Docs by LangChain


## Bước 3: Embedding + lưu vào FAISS

In [4]:
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)

os.makedirs(INDEX_DIR, exist_ok=True)
vectorstore.save_local(INDEX_DIR)
print("Vectors in index:", vectorstore.index.ntotal)

Vectors in index: 37


## Bước 4: Xuất index.json để xem chunk
(Tuỳ chọn) Cell cuối của bước này có thể load lại index đã lưu để bỏ qua bước 1-3.

In [6]:
metadata = {"chunks": []}
for i in range(vectorstore.index.ntotal):
    doc_id = vectorstore.index_to_docstore_id[i]
    doc = vectorstore.docstore.search(doc_id)
    metadata["chunks"].append({
        "id": str(doc_id),
        "text": doc.page_content,
        "source": doc.metadata.get("source", ""),
        "metadata": doc.metadata,
    })

with open(f"{INDEX_DIR}/index.json", "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)
print("Exported index.json")

Exported index.json


In [ ]:
# (Tuỳ chọn) Load lại index đã lưu, bỏ qua bước 1-3
# vectorstore = FAISS.load_local(
#     INDEX_DIR, OpenAIEmbeddings(), allow_dangerous_deserialization=True
# )

## Bước 5: Retrieve - tìm chunk liên quan

In [7]:
query = "What is LangChain?"  # đổi câu hỏi tại đây

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i + 1}:")
    print(doc.page_content[:500])


Document 1:
Quickstart - Docs by LangChain

Document 2:
Continue with:

LangChain agents: Add and manage memory, deploy to production
Deep Agents: Customization options, persistent memory, deploy to production


Connect these docs to your agent of choice via MCP for real-time answers.Edit this page on GitHub or file an issue.Was this page helpful?YesNoInstall LangChainPreviousChangelogNextDocs by LangChain home pagegithubxlinkedinyoutubeResourcesForumChangelogLangChain AcademyContact SalesCompanyHomeTrust CenterCareersBloggithubxlinkedinyoutube

Document 3:
Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationGet startedQuickstartOverviewManaged Deep AgentsDee

## Bước 6: Generate - LLM trả lời dựa trên chunk tìm được

In [8]:
chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True,
)

response = chain.invoke({"query": query})
print("Sources:", len(response["source_documents"]))
print("Response:", response["result"])

Sources: 3
Response: LangChain is a framework designed for developing applications that utilize language models. It provides tools and components to build, manage, and deploy language model applications, including features for memory management, agent customization, and integration with various APIs and services. LangChain aims to simplify the process of creating sophisticated applications that leverage natural language processing capabilities.
